# Learning Rate Scheduler

learning rate scheduler는 학습 중 learning rate를 자동으로 조절하는 도구이다.

- learning rate가 너무 크면 학습이 흔들릴 수 있다.
- learning rate가 너무 작으면 학습이 느릴 수 있다.
- 그래서 학습 중간에 learning rate를 조절하는 scheduler를 함께 사용하는 경우가 많다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.set_num_threads(1)
torch.manual_seed(42)
np.random.seed(42)

## 1. 데이터 준비

In [ ]:
# 데이터 생성
X, y = make_moons(n_samples=1000, noise=0.25, random_state=42)

# train / validation 분리
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 입력값 스케일 정리
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# tensor 변환
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
y_val = torch.tensor(y_val.reshape(-1, 1), dtype=torch.float32)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)

## 2. 모델 및 평가 함수 준비

In [ ]:
class BinaryMLP(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def evaluate_binary_model(model, X_train, y_train, X_val, y_val):
    criterion = nn.BCEWithLogitsLoss()

    model.eval()
    with torch.no_grad():
        train_logits = model(X_train)
        val_logits = model(X_val)

        train_loss = criterion(train_logits, y_train).item()
        val_loss = criterion(val_logits, y_val).item()

        train_pred = (torch.sigmoid(train_logits) >= 0.5).float()
        val_pred = (torch.sigmoid(val_logits) >= 0.5).float()

        train_acc = (train_pred == y_train).float().mean().item()
        val_acc = (val_pred == y_val).float().mean().item()

    return train_loss, val_loss, train_acc, val_acc

## 3. StepLR

`StepLR`은 일정 epoch마다 learning rate를 줄인다.

주요 파라미터
- `step_size`: 몇 epoch마다 learning rate를 줄일지
- `gamma`: 줄일 비율. 예를 들어 `0.5`이면 절반으로 감소

즉
- `step_size=30`
- `gamma=0.5`

이면 30 epoch마다 learning rate가 절반이 된다.

In [ ]:
def train_without_scheduler(initial_lr=1.0, epochs=120):
    torch.manual_seed(42)

    model = BinaryMLP(hidden_size=64)
    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.SGD(model.parameters(), lr=initial_lr, momentum=0.9)

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    lr_history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        train_logits = model(X_train)
        train_loss = criterion(train_logits, y_train)

        train_loss.backward()
        optimizer.step()

        train_loss_eval, val_loss_eval, train_acc, val_acc = evaluate_binary_model(
            model, X_train, y_train, X_val, y_val
        )

        train_losses.append(train_loss_eval)
        val_losses.append(val_loss_eval)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        lr_history.append(optimizer.param_groups[0]['lr'])

    return train_losses, val_losses, train_accs, val_accs, lr_history


## 4. ReduceLROnPlateau

`ReduceLROnPlateau`는 validation loss 같은 지표가
한동안 좋아지지 않을 때 learning rate를 줄이는 방식이다.

주요 파라미터
- `mode='min'`: loss처럼 작을수록 좋은 값을 볼 때 사용
- `patience`: 몇 epoch 동안 개선이 없으면 줄일지
- `factor`: 줄일 비율
- `threshold`: 얼마나 좋아져야 **개선** 으로 볼지 기준
- `min_lr`: learning rate가 너무 작아지지 않도록 하한선 지정

즉 StepLR이 미리 정한 규칙이라면,
ReduceLROnPlateau는 실제 검증 지표 반응에 따라 움직이는 방식이다.

## 5. CosineAnnealingLR

`CosineAnnealingLR`은 learning rate를
코사인 곡선 형태로 부드럽게 줄여가는 방식이다.

주요 파라미터
- `T_max`: 한 주기 길이
- `eta_min`: 내려갈 learning rate의 최솟값

계단처럼 뚝 줄이기보다
완만하게 감소시키는 흐름을 보고 싶을 때 자주 사용한다.

## 6. OneCycleLR

`OneCycleLR`은 학습 초반에 learning rate를 올렸다가
이후에는 다시 크게 낮추는 방식이다.

주요 파라미터
- `max_lr`: 올라갔을 때의 최고 learning rate
- `epochs`: 전체 epoch 수
- `steps_per_epoch`: epoch당 optimizer step 횟수
- `pct_start`: 초반 몇 퍼센트 구간까지 learning rate를 올릴지

중요한 점은
`OneCycleLR`은 보통 batch마다 `scheduler.step()`을 호출한다는 것이다.
즉 epoch마다 한 번이 아니라 mini-batch마다 한 번씩 반영되는 방식이다.

## 정리

1. scheduler는 학습 중 learning rate를 자동으로 조절하는 도구이다.
2. StepLR은 일정 epoch마다 learning rate를 줄인다.
3. ReduceLROnPlateau는 validation loss 같은 지표가 정체될 때 줄인다.
4. CosineAnnealingLR은 learning rate를 부드러운 곡선 형태로 줄인다.
5. OneCycleLR은 초반에 learning rate를 올리고 이후에는 크게 낮춘다.
6. scheduler를 이해할 때는 언제 줄이는가와 `step()`을 어디서 호출하는가를 함께 보는 것이 중요하다.